# HiBASIL Tutorial 1: Finding the Needle in the Haystack

This notebook introduces the HiBASIL (BAyesian Source Inference and Localization) framework. We will simulate a disease outbreak from an unknown source and use Bayesian inference to recover the source location and dispersal parameters.

## 1. Environment Setup
First, we prepare the workspace by creating directories and loading the necessary Bayesian engines. 

In [1]:
import os
import numpy as np
import pandas as pd
# Set environment flags for PyTensor
os.environ['PYTENSOR_FLAGS'] = "base_compiledir=./pytensor_cache,cxx="

import pymc as pm
import arviz as az
from tqdm.auto import tqdm

import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score

# Create necessary directories automatically
for folder in ['./sim', './output', './pytensor_cache']:
    os.makedirs(folder, exist_ok=True)

import simulation  # simulation.py developed in this study
import hibasil       # core functions of the HiBASIL framework

np.random.seed(42)

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
C:\Users\sunny\anaconda3\envs\mcenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ PyTensor cache: pytensor_cache


## 2. Generating the "Mystery" Data
We simulate a single-focus outbreak using a Power-Law dispersal kernel. Note the use of eps = 1e-12 to ensure numerical stability.

In [2]:
# Simulation Constants
eps = 1e-12 
model_type = 'power_law'
true_focus = [0.0, 0.0, 0.6] # [X, Y, Intensity]
true_scale = 5.0
true_exponent = 2.0
p0 = 0.3 # baseline probability of zero
q_param = 0.01 # probability of one
phi = 18 # precision

# Define sampling coordinates (Transect)
y_coords = np.linspace(-2, 50, 250)
y_coords = np.unique(np.concatenate([[true_focus[1]], y_coords]))
x_coords = np.zeros_like(y_coords)
sample_points = np.column_stack((x_coords, y_coords))

# Generate simulated disease severity using the ZOIB framework
sample_kernel = simulation.single_focus_kernel(sample_points[:,0], sample_points[:, 1], true_focus, true_scale, model_type=model_type, exponent=true_exponent)
p_param = np.clip(p0 * (1- sample_kernel), eps, 1- eps) # probability of zero at each sampling location
mu = np.clip(sample_kernel, eps, 1- eps) # mean disease severity at each sampling location 

alpha_samples = mu * phi # calculate parameter alpha of Beta distribution
beta_samples = (1 - mu) * phi # calculate parameter beta of Beta distribution

prob_zero = p_param  # probability of structural zero
prob_one = (1.0 - p_param) * q_param # probability of structural zero
prob_continuous = (1.0 - p_param) * (1.0 - q_param)   # calculate probability of continous severity

n_rows = 4
obs_df = pd.DataFrame({
    'InterrowDistance': pd.Series(dtype='float64'),
    'Distance': pd.Series(dtype='float64'),
    'Severity': pd.Series(dtype='float64'),
    'Row': pd.Series(dtype='int64'),
    'Plant': pd.Series(dtype='object')
})
for run_id in range(n_rows):
    y_obs_run = simulation.sampling(sample_points, true_focus, prob_zero, prob_one, prob_continuous, alpha_samples, beta_samples)
    row_val = run_id%n_rows
    row_label = np.full(len(sample_points), row_val)
    y_obs_run['Row'] = row_label
    
    obs_df = pd.concat([obs_df, y_obs_run])        

print(obs_df.head())
obs_df.to_csv("./sim/obs_df.csv")


# Visualize the initial state
plt.figure(figsize=(10, 6))

# Plot each row with a different color to verify the simulation logic
for row_id in obs_df['Row'].unique():
    row_data = obs_df[obs_df['Row'] == row_id]
    plt.scatter(row_data['Distance'], row_data['Severity'], alpha=0.4, label=f'Row {row_id}')

# Mark the true focus clearly
plt.axvline(true_focus[1], color='red', linestyle='--', linewidth=2, label='True Focus (Source)')

plt.title("Simulated Disease Gradient (Raw Data)")
plt.xlabel("Distance from Transect Start")
plt.ylabel("Disease Severity (0 to 1)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('./output/disease_gradient.png')
plt.show()

   InterrowDistance  Distance  Severity  Row Plant
0               0.0 -2.000000  0.185855    0     0
1               0.0 -1.791165  0.408659    0     1
2               0.0 -1.582329  0.320431    0     2
3               0.0 -1.373494  0.238264    0     3
4               0.0 -1.164659  0.503869    0     4


## 3. The HiBASIL Model:Priors and Likelihood
We use weakly informed priors to guide the model without forcing a specific answer.
Coordinates ($f_x$, $f_y$): Normal distribution centered on the field average.
Intensity ($f_z$): Beta (1,1) for a flat unbiased start.
Likelihood: Zero-Inflated Beta (ZOIB) to handle healthy plants and complete infections simultaneously.

In [3]:
model = hibasil.build_single_focus_model(obs_df, model_type=model_type)

with model:
    print("\n--- Starting MCMC Sampling ---")
    trace = pm.sample(draws=4000, tune=2000, chains=2, target_accept=0.95, random_seed=42, cores=2, nuts_sampler="nutpie")
    
    # Generate Posterior Predictive Check (PPC)
    thin_trace = trace.sel(draw=slice(None, None, 5))  # use every 5th draw
    ppc = pm.sample_posterior_predictive(thin_trace, var_names=['obs'], progressbar=True, random_seed=42)

    # plot posterior predictive check
    y_sim = ppc.posterior_predictive['obs'].values.astype(np.float64)                
    y_obs = obs_df['Severity'].values
    distances = obs_df['Distance'].values
                
    hibasil.plot_posterior_predictive(y_sim, y_obs, distances, model_type, sim=0)                
                        
# calculate ppc metrics
y_pred = ppc.posterior_predictive['obs'].mean(dim=['chain', 'draw']).values
output_file = './output/ppc_metrics_all_sims.csv'
pd.DataFrame(columns=['simulation', 'model_type', 'overall_r2', 'overall_rmse', 'overall_mae',
                        'obs_zeros', 'pred_zeros', 'obs_ones', 'pred_ones', 'continuous_r2', 
                        'continuous_rmse']).to_csv(output_file, index=False) # store all ppc metrics
            
hibasil.compute_and_save_metrics(y_obs, y_pred, y_sim, model_type, 0, output_file)


Multi-row model preparation:
- Total observations: 1004
- Number of rows: 4
- Rows: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
- Severity range: [0.0, 1.0]
- Exact zeros: 299 (29.8%)
- Exact ones: 5 (0.5%)
- Continuous: 700 (69.7%)

--- Starting MCMC Sampling ---


Progress,Draws,Divergences,Step Size,Gradients/Draw
,6000,14,0.13,31
,6000,42,0.17,31


Sampling: [obs]


{'simulation': 0,
 'model_type': 'power_law',
 'overall_r2': 0.5043405271944807,
 'overall_rmse': np.float64(0.09490453587034774),
 'overall_mae': 0.04042803861114527,
 'obs_zeros': np.float64(0.29780876494023906),
 'pred_zeros': np.float64(0.29881225099601594),
 'obs_ones': np.float64(0.0049800796812749),
 'pred_ones': np.float64(0.005726469123505976),
 'continuous_r2': 0.8445804797650085,
 'continuous_rmse': np.float64(0.05295509728619202)}

## 4. Results and Spatial Visualization

We evaluate the model's accuracy by mapping the estimated focus against the true coordinates.

In [4]:
# 1. Check Convergence (R-hat should be <= 1.01)
summary = az.summary(trace, var_names=['fx1', 'fy1', 'fz1', 'scale1', 'exponent1'])
print(summary[['mean', 'hdi_3%', 'hdi_97%', 'r_hat']])

# 2. Universal 2D Comparison Plot
# Provides a visual 'Needle in a Haystack' confirmation
obs_data, focus_coords_dict = basil.load_and_process_multi_row_data(obs_df)
hibasil.plot_multi_row_results(trace, obs_data, focus_coords_dict, y_pred, model_type, sim=0)

print()
print("Congratulations! You have finished Tutorial 1!")

            mean  hdi_3%  hdi_97%  r_hat
fx1[0]     0.044  -0.804    0.880   1.00
fx1[1]     0.149  -1.255    1.380   1.01
fx1[2]     0.070  -0.959    1.025   1.00
fx1[3]     0.042  -0.720    0.792   1.00
fy1[0]    -0.080  -0.386    0.223   1.00
fy1[1]     0.256  -0.171    0.693   1.00
fy1[2]     0.158  -0.185    0.526   1.00
fy1[3]     0.140  -0.169    0.465   1.00
fz1[0]     0.626   0.536    0.725   1.00
fz1[1]     0.632   0.536    0.749   1.00
fz1[2]     0.653   0.556    0.772   1.00
fz1[3]     0.634   0.547    0.732   1.00
scale1     5.233   3.650    6.852   1.00
exponent1  2.150   1.888    2.416   1.00
Foci locations: 4 points
Observations: 1000 points
Unique rows: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
 Row 0: Found focus1 at (0.0, 0.0)
 Row 1: Found focus1 at (0.0, 0.0)
 Row 2: Found focus1 at (0.0, 0.0)
 Row 3: Found focus1 at (0.0, 0.0)

Congratulations! You have finished Tutorial 1!
